# DSPy Optimization — BootstrapFewShot vs MIPROv2

**Week 6 | Notebook 2 of 6**

**What you'll learn:**
- What is compilation? (conceptual walkthrough)
- Preparing trainset + devset (dspy.Example format)
- Writing a custom metric function
- Running BootstrapFewShot — inspect generated few-shot demos
- Running MIPROv2 — inspect generated instructions
- Comparing: baseline vs optimized on devset score
- Saving and reloading the optimized program

**Runtime:** ~45 minutes (API calls during optimization)

**Cost-saving:** Default 10 trials (reduce in .env with DSPY_OPTIMIZER_TRIALS)

In [1]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("06_dspy/02_optimizers.ipynb")

💰 COST ESTIMATE
----------------------------------------
Notebook:  06_dspy/02_optimizers.ipynb
Task:      DSPy optimizers (expensive)
Calls:     ~60

With GPT-4o:       $0.90 USD
With GPT-4o-mini:  $0.09 USD (10x cheaper)
With Ollama:       $0.00 USD (free, local)

💡 TIP: Set USE_SMALL_MODEL=true or USE_OLLAMA=true in .env to save money.
----------------------------------------


## 1. Setup

In [2]:
import dspy
from dspy.evaluate import Evaluate
from dspy.teleprompt import BootstrapFewShot, MIPROv2

from src.config import DSPY_OPTIMIZER_TRIALS, get_dspy_lm
from src.datasets import generate_qa_pairs

lm = get_dspy_lm()
dspy.configure(lm=lm)

print("✅ DSPy configured")
print(f"Optimizer trials: {DSPY_OPTIMIZER_TRIALS}")

✅ DSPy configured
Optimizer trials: 10


## 2. What Is Compilation?

In [3]:
# DSPy compilation = automatically improving your program
# by generating better instructions and few-shot examples

print("DSPy Compilation Flow:")
print("  1. Define your program (signatures + modules)")
print("  2. Provide training examples")
print("  3. Define a metric (0-1 scoring function)")
print("  4. Run optimizer (BootstrapFewShot / MIPROv2)")
print("  5. Optimized program has better prompts + demos")
print("  6. Evaluate on devset")
print("\n💡 Think of it like a compiler: Python → optimized prompts")

DSPy Compilation Flow:
  1. Define your program (signatures + modules)
  2. Provide training examples
  3. Define a metric (0-1 scoring function)
  4. Run optimizer (BootstrapFewShot / MIPROv2)
  5. Optimized program has better prompts + demos
  6. Evaluate on devset

💡 Think of it like a compiler: Python → optimized prompts


## 3. Preparing Trainset + Devset

In [4]:
# Convert synthetic data to dspy.Example format
qa_data = generate_qa_pairs(40)

examples = [
    dspy.Example(question=d["question"], answer=d["expected"]).with_inputs("question")
    for d in qa_data
]

# Split: 70% train, 30% dev
split = int(len(examples) * 0.7)
trainset = examples[:split]
devset = examples[split:]

print(f"Trainset: {len(trainset)} examples")
print(f"Devset: {len(devset)} examples")
print("\nSample train example:")
print(f"  Question: {trainset[0].question}")
print(f"  Answer: {trainset[0].answer}")

Trainset: 28 examples
Devset: 12 examples

Sample train example:
  Question: What is DSPy?
  Answer: DSPy is a framework for programming language models.


## 4. Writing a Custom Metric

In [5]:
def answer_metric(example, prediction, trace=None):
    """Check if predicted answer covers the expected answer's content.

    Word-overlap ratio (expected words found in prediction). A plain substring
    check scores 0 on paraphrases, which leaves optimizers with nothing to
    bootstrap from.
    """
    expected = set(example.answer.lower().split())
    predicted = set(prediction.answer.lower().split())
    if not expected:
        return 0.0
    return 1.0 if len(expected & predicted) / len(expected) >= 0.5 else 0.0


# Test the metric
class TestEx(dspy.Example):
    pass


ex = dspy.Example(question="What is RAG?", answer="Retrieval-Augmented Generation").with_inputs(
    "question"
)
pred = dspy.Prediction(answer="RAG stands for Retrieval-Augmented Generation")
print(f"Metric score: {answer_metric(ex, pred)}")

Metric score: 1.0


## 5. Baseline Program

In [6]:
class QA(dspy.Signature):
    """Answer questions with short factual responses."""

    question: str = dspy.InputField()
    answer: str = dspy.OutputField()


class SimpleQA(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate = dspy.ChainOfThought(QA)

    def forward(self, question):
        return self.generate(question=question)


baseline = SimpleQA()

# Evaluate baseline
evaluator = Evaluate(devset=devset, metric=answer_metric, num_threads=4, display_progress=True)
baseline_score = evaluator(baseline).score
print(f"\nBaseline score: {baseline_score:.2f}")

  0%|          | 0/12 [00:00<?, ?it/s]

Average Metric: 1.00 / 1 (100.0%):   0%|          | 0/12 [00:01<?, ?it/s]

Average Metric: 1.00 / 1 (100.0%):   8%|▊         | 1/12 [00:01<00:20,  1.86s/it]

Average Metric: 1.00 / 2 (50.0%):   8%|▊         | 1/12 [00:01<00:20,  1.86s/it] 

Average Metric: 2.00 / 3 (66.7%):  17%|█▋        | 2/12 [00:01<00:18,  1.86s/it]

...

Average Metric: 7.00 / 11 (63.6%):  83%|████████▎ | 10/12 [00:02<00:00,  3.24it/s]

Average Metric: 7.00 / 12 (58.3%):  92%|█████████▏| 11/12 [00:02<00:00,  3.24it/s]

Average Metric: 7.00 / 12 (58.3%): 100%|██████████| 12/12 [00:02<00:00,  5.88it/s]

2026/09/18 20:16:41 INFO dspy.evaluate.evaluate: Average Metric: 7.0 / 12 (58.3%)




Baseline score: 58.33


## 6. BootstrapFewShot — Quick Baseline Optimizer

In [7]:
# BootstrapFewShot: generates few-shot demos by running the program
teleprompter = BootstrapFewShot(metric=answer_metric, max_bootstrapped_demos=4)

optimized_bootstrap = teleprompter.compile(SimpleQA(), trainset=trainset)

# Evaluate optimized
bootstrap_score = evaluator(optimized_bootstrap).score
print(f"\nBootstrapFewShot score: {bootstrap_score:.2f}")
print(f"Improvement: {bootstrap_score - baseline_score:+.2f}")

  0%|          | 0/28 [00:00<?, ?it/s]

 32%|███▏      | 9/28 [00:00<00:00, 88.61it/s]

 50%|█████     | 14/28 [00:00<00:00, 91.91it/s]

Bootstrapped 4 full traces after 14 examples for up to 1 rounds, amounting to 14 attempts.


  0%|          | 0/12 [00:00<?, ?it/s]

...

Average Metric: 8.00 / 12 (66.7%):  92%|█████████▏| 11/12 [00:04<00:00,  4.31it/s]

Average Metric: 8.00 / 12 (66.7%): 100%|██████████| 12/12 [00:04<00:00,  2.23it/s]

Average Metric: 8.00 / 12 (66.7%): 100%|██████████| 12/12 [00:04<00:00,  2.45it/s]

2026/09/18 20:16:47 INFO dspy.evaluate.evaluate: Average Metric: 8.0 / 12 (66.7%)




BootstrapFewShot score: 66.67
Improvement: +8.34


## 7. MIPROv2 — Bayesian Optimization

In [8]:
# MIPROv2: Bayesian optimization of instructions + demos
# More expensive but generally better results

mipro = MIPROv2(
    metric=answer_metric,
    auto=None,  # dspy 3.3: opt out of auto budget to set candidates/trials manually
    num_candidates=5,  # Reduced for cost
    init_temperature=1.0,
)

optimized_mipro = mipro.compile(
    SimpleQA(),
    trainset=trainset,
    num_trials=DSPY_OPTIMIZER_TRIALS,  # Configurable via .env
    valset=devset,
    minibatch=False,  # Small valset (12) — default minibatch size is 35
)

# Evaluate optimized
mipro_score = evaluator(optimized_mipro).score
print(f"\nMIPROv2 score: {mipro_score:.2f}")
print(f"Improvement over baseline: {mipro_score - baseline_score:+.2f}")
print(f"Improvement over Bootstrap: {mipro_score - bootstrap_score:+.2f}")

2026/09/18 20:16:47 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==


2026/09/18 20:16:47 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.



2026/09/18 20:16:47 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=5 sets of demonstrations...


Bootstrapping set 1/5
Bootstrapping set 2/5
Bootstrapping set 3/5


  0%|          | 0/28 [00:00<?, ?it/s]

...

Average Metric: 9.00 / 11 (81.8%):  83%|████████▎ | 10/12 [00:00<00:00, 188.88it/s]

Average Metric: 9.00 / 12 (75.0%):  92%|█████████▏| 11/12 [00:00<00:00, 191.70it/s]

Average Metric: 9.00 / 12 (75.0%): 100%|██████████| 12/12 [00:00<00:00, 207.86it/s]

2026/09/18 20:18:07 INFO dspy.evaluate.evaluate: Average Metric: 9.0 / 12 (75.0%)




MIPROv2 score: 75.00
Improvement over baseline: +16.67
Improvement over Bootstrap: +8.33


## 8. Inspecting Generated Instructions

In [9]:
# Inspect what the optimizer generated
print("Generated demos (BootstrapFewShot):")
demos = getattr(optimized_bootstrap.generate, "demos", [])  # absent if none kept
print(f"  {len(demos)} demos")
if not demos:
    print("  (No demos passed the metric — a looser metric yields more demos.)")
for i, demo in enumerate(demos[:2]):
    print(f"\n  Demo {i + 1}:")
    print(f"    Question: {demo.question}")
    print(f"    Answer: {demo.answer}")

print("\n" + "=" * 50)
print("Generated instructions may be embedded in the compiled program.")
print("Use optimized_mipro.save() to inspect the full program.")

Generated demos (BootstrapFewShot):
  0 demos
  (No demos passed the metric — a looser metric yields more demos.)

Generated instructions may be embedded in the compiled program.
Use optimized_mipro.save() to inspect the full program.


## 9. Saving and Reloading

In [10]:
# Save optimized program
optimized_mipro.save("optimized_qa.json")
print("✅ Saved to optimized_qa.json")

# Reload
loaded = SimpleQA()
loaded.load("optimized_qa.json")

# Verify it works
result = loaded(question="What is DSPy?")
print(f"\nReloaded program answer: {result.answer}")

✅ Saved to optimized_qa.json

Reloaded program answer: DSPy is a framework that uses domain-specific programming languages to simplify AI model development.


## 10. Exercise: Optimize a Classification Pipeline

Optimize a sentiment classifier on your own dataset using:
1. BootstrapFewShot as baseline
2. MIPROv2 for best results
3. Compare scores and inspect generated demos

In [11]:
# YOUR TURN: Optimize a classification pipeline

# class Sentiment(dspy.Signature):
#     """Classify sentiment."""
#     text: str = dspy.InputField()
#     sentiment: str = dspy.OutputField()

# class SentimentClassifier(dspy.Module):
#     def __init__(self):
#         super().__init__()
#         self.classify = dspy.ChainOfThought(Sentiment)
#     def forward(self, text):
#         return self.classify(text=text)

# # Optimize
# teleprompter = MIPROv2(metric=your_metric)
# optimized = teleprompter.compile(SentimentClassifier(), trainset=trainset, num_trials=10)

---

**Next:** [03_rag_pipeline.ipynb](03_rag_pipeline.ipynb) — RAG with assertions and constraints